# ndvi2gif: de las imágenes a la serie temporal

**Curso avanzado de SIG en ecología · EBD-CSIC · septiembre de 2026**

[ndvi2gif](https://github.com/Digdgeo/Ndvi2Gif) es una librería de Python que trabaja sobre **Google Earth Engine**. Hace una cosa muy concreta y muy útil: coge miles de imágenes de satélite y las resume en **composites por periodo** (estaciones, meses...) para cada año. Con eso se hacen mapas, GIFs, series temporales y tendencias.

En este notebook vamos a recorrer lo que necesitaréis el viernes:

| Parte | Qué hacemos | Dónde |
|---|---|---|
| 1 | Conectar con Earth Engine | — |
| 2 | Primer composite estacional con Sentinel-2 | Marisma de Doñana |
| 3 | Estadísticas por recinto y un GIF | Marisma de Doñana |
| 4 | Una zona definida con un modelo digital de elevaciones | Sierra de Gredos |
| 5 | Serie larga con Landsat (1985–2025) y su tendencia | Sierra de Gredos |
| 6 | Mapa de tendencia píxel a píxel | Sierra de Gredos |
| 7 | **Trampas**: lo que puede fabricar una tendencia falsa | Sierra de Gredos |
| 8 | Guardar resultados | — |

> **Antes de empezar:** necesitas tu cuenta de Earth Engine con un **proyecto de Google Cloud registrado** (lo hicimos el miércoles). Ten a mano el nombre del proyecto, algo como `ee-tunombre`.

## 1. Conectar con Earth Engine

En Colab hay que instalar ndvi2gif cada vez que se abre una sesión nueva. `geemap` (mapas interactivos) ya viene instalado en Colab.

In [ ]:
%pip install -q ndvi2gif

La primera vez, `ee.Authenticate()` abre una ventana para dar permiso con tu cuenta de Google. **Cambia `PROYECTO` por el tuyo.**

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
from ndvi2gif import NdviSeasonality, TimeSeriesAnalyzer

PROYECTO = 'ee-tunombre'   # <-- ¡cámbialo!

ee.Authenticate()
ee.Initialize(project=PROYECTO)
print('Earth Engine listo:', ee.String('hola').getInfo())

## 2. La idea clave: el composite

Un satélite como Sentinel-2 pasa cada 5 días. En un año son ~70 imágenes, muchas con nubes. ndvi2gif hace esto por nosotros:

1. **Filtra** la colección por zona (`roi`), satélite (`sat`) y años (`start_year` a `end_year`, **ambos incluidos**).
2. **Enmascara** nubes y sombras.
3. **Calcula el índice** (`index`) en cada imagen.
4. **Parte cada año** en `periods` trozos: 4 = estaciones, 12 = meses, 24 = quincenas.
5. **Resume** todas las imágenes de cada trozo con un estadístico (`key`): `'max'`, `'median'`, `'mean'`, `'percentile'`...

El resultado es **una imagen por año**, con **una banda por periodo**.

```
2019 → [winter, spring, summer, autumn]
2020 → [winter, spring, summer, autumn]
...
```

| Parámetro | Ejemplos | Pregunta que responde |
|---|---|---|
| `roi` | `ee.Geometry`, `'s2:T29SQB'`, `'wrs:202,034'` | ¿Dónde? |
| `sat` | `'S2'`, `'Landsat'`, `'MODIS'`, `'S1'` | ¿Con qué sensor? |
| `index` | `'ndvi'`, `'evi'`, `'ndwi'`, `'mndwi'`, `'ndsi'` | ¿Qué variable? |
| `periods` | `4`, `12`, `24` | ¿Cada cuánto? |
| `key` | `'max'`, `'median'`, `'percentile'` | ¿Cómo resumo cada periodo? |
| `start_year`, `end_year` | `2019`, `2024` | ¿Qué años? |

### Nuestra zona: la marisma de Doñana

In [ ]:
marisma_caja = ee.Geometry.Rectangle([-6.50, 36.84, -6.24, 37.16])   # [oeste, sur, este, norte]

donana = NdviSeasonality(
    roi=marisma_caja,
    sat='S2',
    index='ndvi',
    periods=4,
    key='median',
    start_year=2019,
    end_year=2024,
)

`get_year_composite()` construye la colección de composites. Tarda un poco (más o menos un minuto) porque comprueba periodo a periodo que haya datos.

In [ ]:
composites = donana.get_year_composite()

print('Número de años:', composites.size().getInfo())
print('Bandas de cada año:', composites.first().bandNames().getInfo())

### Visualizar: un RGB hecho de estaciones

Un truco clásico: pintar **invierno en rojo, primavera en verde y verano en azul**. El color de cada píxel cuenta su calendario:

- **Blanco/gris claro:** verde todo el año (pinar, matorral denso).
- **Rojo:** verde solo en invierno.
- **Amarillo** (rojo + verde): verde en invierno y primavera, seco en verano.
- **Azul:** verde solo en verano (por ejemplo, arrozales).
- **Negro:** nunca verde (agua, arena, suelo desnudo).

Busca en el mapa ejemplos de cada color. ¿Qué es cada cosa?

In [ ]:
vis = {'bands': ['winter', 'spring', 'summer'], 'min': 0.1, 'max': 0.8}

m = geemap.Map(center=[37.0, -6.37], zoom=10)
m.add_basemap('SATELLITE')
m.addLayer(composites.median(), vis, 'NDVI estacional (mediana 2019-2024)')

# Un único año: filtramos la colección por fecha
anio_2023 = composites.filterDate('2023-01-01', '2023-12-31').first()
m.addLayer(anio_2023, vis, 'NDVI estacional 2023', False)
m

### ✏️ Prueba

1. Cambia `index='ndvi'` por `index='mndwi'` (índice de agua) y usa `vis = {'bands': ['winter', 'spring', 'summer'], 'min': -0.5, 'max': 0.5}`. ¿Qué zonas se inundan en invierno?
2. Cambia `key='median'` por `key='max'`. ¿Qué cambia en el mapa? ¿Por qué el máximo suele verse "más verde"?
3. ¿Qué índices hay disponibles para cada satélite?

In [ ]:
print('Landsat:', donana.get_available_indices('Landsat'))

## 3. Estadísticas por recinto y un GIF

### Traer un fichero vectorial a Earth Engine

Usamos los **recintos de la marisma** del ejercicio de GeoLibre. Los leemos con GeoPandas (lo vimos en la intro de Python) y los convertimos a objetos de Earth Engine.

> ndvi2gif también acepta directamente la ruta de un `.shp` o un `.geojson` como `roi`. Pero los GeoJSON escritos por GeoPandas llevan un `id` numérico en cada entidad y Earth Engine lo rechaza (*"system:index must be a string"*). Pasando por GeoPandas nos lo ahorramos.

In [ ]:
import geopandas as gpd

URL_DATOS = 'https://raw.githubusercontent.com/Digdgeo/sig-avanzado-2026/main/datos'
recintos = gpd.read_file(f'{URL_DATOS}/ligero/recintos_marisma.geojson')
recintos

In [ ]:
# De GeoDataFrame a ee.FeatureCollection
recintos_ee = ee.FeatureCollection(recintos.to_crs(4326).__geo_interface__)

# Mediana del NDVI de cada estación, en cada recinto, para 2023
stats = anio_2023.reduceRegions(
    collection=recintos_ee,
    reducer=ee.Reducer.median(),
    scale=20,
)
tabla = ee.data.computeFeatures({'expression': stats, 'fileFormat': 'PANDAS_DATAFRAME'})
tabla = tabla[['recinto', 'winter', 'spring', 'summer', 'autumn']].round(3)
tabla

In [ ]:
tabla.set_index('recinto').T.plot(marker='o', figsize=(8, 4), title='NDVI mediano por estación, 2023')
plt.ylabel('NDVI')
plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()

### Un GIF con la evolución de los años

`get_gif()` genera un fotograma por año con las tres bandas que elijas. Se guarda en la carpeta de Colab (icono de carpeta a la izquierda). La versión `_texted` lleva el año escrito. Tarda un par de minutos, porque vuelve a calcular los composites.

In [ ]:
donana.get_gif(name='donana_s2.gif', bands=['winter', 'spring', 'summer'])

In [ ]:
from IPython.display import Image
Image(filename='donana_s2_texted.gif')

## 4. Hacia el viernes: una zona definida por la altitud

El viernes trabajaremos en **alta montaña**. Allí la zona de estudio no es un rectángulo, sino **todo lo que está por encima de cierta cota**. Aquí practicamos la receta en la **Sierra de Gredos** (Almanzor, 2.591 m), con el umbral de **2.000 m**.

Los pasos son:

1. Modelo digital de elevaciones Copernicus GLO-30 (versión 2024).
2. Umbral: `altitud >= 2000`.
3. Vectorizar el resultado a 90 m, para que sea ligero.

In [ ]:
dem = ee.ImageCollection('COPERNICUS/DEM/GLO30_2024_1').select('DEM').mosaic()

caja_gredos = ee.Geometry.Rectangle([-5.45, 40.18, -5.00, 40.32])
COTA = 2000

cumbres = (dem.gte(COTA).selfMask().clip(caja_gredos)
           .reduceToVectors(geometry=caja_gredos, scale=90, geometryType='polygon',
                            eightConnected=True, maxPixels=1e9)
           .geometry(maxError=30))

print(f'Superficie por encima de {COTA} m: {cumbres.area(10).divide(1e6).getInfo():.1f} km²')

In [ ]:
m2 = geemap.Map(center=[40.25, -5.25], zoom=11)
m2.add_basemap('SATELLITE')
m2.addLayer(dem.clip(caja_gredos), {'min': 1000, 'max': 2600, 'palette': ['006837', 'fee08b', 'ffffff']}, 'Altitud', False)
m2.addLayer(ee.FeatureCollection([ee.Feature(cumbres)]).style(color='ff0000', fillColor='ff000033'), {}, f'Por encima de {COTA} m')
m2

### ✏️ Prueba

Cambia `COTA` a 1.800 y a 2.200 m. ¿Cómo cambia la superficie? ¿Qué cota tiene sentido para hablar de "cumbres"?

## 5. Serie larga con Landsat: 41 veranos

Landsat es el único programa con imágenes desde **1984**. ndvi2gif **fusiona Landsat 4, 5, 7, 8 y 9** en una sola colección.

Nos quedamos con el **máximo de NDVI del verano** (julio–septiembre), que es cuando la montaña está sin nieve y en plena actividad.

In [ ]:
gredos = NdviSeasonality(
    roi=cumbres,
    sat='Landsat',
    index='ndvi',
    periods=4,             # winter, spring, summer, autumn
    key='max',
    start_year=1985,
    end_year=2025,
    max_cloud_cover=60,    # en montaña hay muchas nubes: somos menos exigentes
)
print(gredos.period_names)
print(gredos.period_dates)

### ⚡ El truco para que no tarde 15 minutos

Podríamos usar `get_year_composite()` como antes, pero con 41 años × 4 estaciones hace **164 consultas** a Earth Engine, una detrás de otra.

La forma rápida es **montar todo el cálculo en el servidor y pedir el resultado una sola vez**:

1. `get_period_composite(año, 2)` nos da el composite de verano (el periodo 2) de un año **sin calcular nada todavía**.
2. Con eso montamos una colección de 41 imágenes.
3. Con `map` + `reduceRegion` calculamos la mediana en la zona de cada imagen.
4. Un único `getInfo()` al final lanza el cálculo.

**Regla de oro de Earth Engine:** evita los `getInfo()` dentro de bucles.

In [ ]:
ANIOS = list(range(1985, 2026))
VERANO = 2   # índice del periodo: 0 winter, 1 spring, 2 summer, 3 autumn

def composite_verano(anio):
    img = gredos.get_period_composite(anio, VERANO)
    return img.set('anio', anio)

veranos = ee.ImageCollection([composite_verano(a) for a in ANIOS])

def ndvi_en_zona(img):
    valor = img.reduceRegion(reducer=ee.Reducer.median(), geometry=cumbres,
                             scale=60, maxPixels=1e9, bestEffort=True)
    # Si un verano no tiene imágenes, el composite no tiene bandas: -999 marca el hueco
    return ee.Feature(None, {'anio': img.get('anio'), 'ndvi': valor.get('nd', -999)})

resultado = ee.FeatureCollection(veranos.map(ndvi_en_zona)).getInfo()   # <- la única llamada

serie = pd.DataFrame([f['properties'] for f in resultado['features']]).sort_values('anio').reset_index(drop=True)
serie['ndvi'] = serie['ndvi'].where(serie['ndvi'] > -1)   # -999 -> NaN (año sin datos)
print('Años sin dato:', serie.loc[serie['ndvi'].isna(), 'anio'].tolist())
serie.head()

Añadimos **cuántas escenas** entran en cada verano. Luego veremos por qué importa.

In [ ]:
ini, fin = gredos.period_dates[VERANO]
n_escenas = ee.List([gredos.ndvi_col.filterDate(f'{a}{ini}', f'{a}{fin}').size() for a in ANIOS]).getInfo()
serie['n_escenas'] = n_escenas

serie.plot(x='anio', y='ndvi', marker='o', figsize=(10, 4), legend=False,
           title='Gredos > 2.000 m · NDVI máximo de verano (mediana de la zona)')
plt.ylabel('NDVI')
plt.grid(alpha=0.3)

### Tendencia: Mann-Kendall y pendiente de Sen

- **Mann-Kendall:** ¿hay una tendencia monótona o puede ser azar? Devuelve **τ** (de −1 a 1) y un **valor p**.
- **Pendiente de Sen:** ¿cuánto cambia? Es la mediana de todas las pendientes entre pares de años, así que **resiste bien los valores atípicos**.

`TimeSeriesAnalyzer.analyze_trend()` necesita un DataFrame con las columnas `date` y `value`.

> Aquí hay **un valor por año**, así que la pendiente sale **por año**. Ojo: el campo `yearly_change` de la regresión lineal supone una serie con `periods` valores por año y aquí no aplica.

In [ ]:
serie['date'] = pd.to_datetime(serie['anio'].astype(str) + '-08-15')
serie['value'] = serie['ndvi']

analizador = TimeSeriesAnalyzer(gredos)
tendencia = analizador.analyze_trend(df=serie.dropna(subset=['value']), method='all')

mk = tendencia['mann_kendall']
sen = tendencia['sen_slope']
print(tendencia['interpretation'])
print(f"τ de Kendall = {mk['tau']:.2f}   ·   p = {mk['p_value']:.2g}")
print(f"Pendiente de Sen = {sen['slope']:+.4f} NDVI/año  →  {sen['slope'] * 10:+.3f} NDVI/década")

## 6. Mapa de tendencia píxel a píxel

La serie resume toda la zona en un número por año. Pero ¿reverdece todo por igual? Earth Engine calcula la **pendiente de Sen en cada píxel** con `ee.Reducer.sensSlope()`. Solo necesita dos bandas: el tiempo (`t`) y la variable (`nd`).

In [ ]:
def con_tiempo(img):
    t = ee.Image.constant(ee.Number(img.get('anio'))).float().rename('t')
    return t.addBands(img.select('nd'))

pendiente = (veranos.map(con_tiempo)
             .reduce(ee.Reducer.sensSlope())
             .select('slope')
             .multiply(10)              # NDVI por década
             .clip(cumbres))

vis_pend = {'min': -0.05, 'max': 0.05, 'palette': ['8c510a', 'd8b365', 'f5f5f5', '5ab4ac', '01665e']}

m3 = geemap.Map(center=[40.25, -5.25], zoom=12)
m3.add_basemap('SATELLITE')
m3.addLayer(pendiente, vis_pend, 'Pendiente de Sen (NDVI/década)')
m3.add_colorbar(vis_pend, label='NDVI / década')
m3

## 7. ⚠️ Trampas: antes de creerse una tendencia

Una pendiente significativa **no demuestra** que la vegetación haya cambiado. En series largas de satélite hay al menos tres formas de fabricar una tendencia falsa. **Estas preguntas son el corazón del ejercicio del viernes.**

### Trampa 1 · El número de escenas y el máximo

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(serie['anio'], serie['ndvi'], 'o-', color='tab:green')
ax1.set_ylabel('NDVI máximo de verano', color='tab:green')
ax2 = ax1.twinx()
ax2.bar(serie['anio'], serie['n_escenas'], alpha=0.25, color='tab:gray')
ax2.set_ylabel('Escenas en el verano', color='tab:gray')
plt.title('¿Sube el NDVI o suben las escenas?')

print('Correlación NDVI ~ nº de escenas:', round(serie['ndvi'].corr(serie['n_escenas'], method='spearman'), 2))

**Piensa:** el **máximo** de 3 imágenes y el máximo de 20 imágenes no son comparables. Con más imágenes es más fácil pillar un día muy verde... o un píxel mal enmascarado. ¿Qué pasaría con `key='median'` o con `key='percentile'` y `percentile=75`?

### Trampa 2 · El cambio de sensor

Landsat 5 y 7 (TM/ETM+) y Landsat 8 y 9 (OLI) **no miden exactamente igual**. ndvi2gif los junta **sin armonizarlos**. OLI tiende a dar un NDVI algo más alto, así que desde 2013 puede aparecer un "escalón" que parece tendencia.

Una forma sencilla de comprobarlo es calcular la tendencia **por épocas**:

In [ ]:
def tendencia_tramo(df, desde, hasta):
    tramo = df[(df['anio'] >= desde) & (df['anio'] <= hasta)].dropna(subset=['value'])
    r = analizador.analyze_trend(df=tramo, method='all')
    return {'tramo': f'{desde}-{hasta}', 'n': len(tramo), 'media': round(tramo['value'].mean(), 3),
            'tau': round(r['mann_kendall']['tau'], 2), 'p': round(r['mann_kendall']['p_value'], 4),
            'sen_decada': round(r['sen_slope']['slope'] * 10, 4)}

pd.DataFrame([
    tendencia_tramo(serie, 1985, 2025),
    tendencia_tramo(serie, 1985, 2011),   # TM / ETM+
    tendencia_tramo(serie, 2013, 2025),   # OLI
])

**Piensa:** ¿la tendencia se mantiene dentro de cada época o todo es el salto entre épocas? ¿Por qué hemos dejado fuera 2012? (Pista: ese año solo volaba Landsat 7, con [el fallo del SLC](https://www.usgs.gov/landsat-missions/landsat-7).)

### Trampa 3 · La nieve

ndvi2gif enmascara nubes y sombras, **pero no la nieve**. Un verano con neveros tardíos tendrá un NDVI más bajo **sin que la vegetación haya cambiado**. Se puede comprobar con otro índice: `index='ndsi'` (nieve) en la misma zona y los mismos veranos.

### ✏️ Retos (elige uno)

1. Repite la serie con `key='median'`. ¿Sigue habiendo tendencia? ¿Y la correlación con el número de escenas?
2. Cambia `VERANO = 2` por la primavera (`1`). ¿Qué cuenta ahora la serie?
3. Crea `NdviSeasonality(..., index='ndsi', key='max')` y calcula la serie de NDSI de verano. ¿Los años con NDVI bajo tienen más nieve?

## 8. Guardar los resultados

Los ficheros de Colab **se borran al cerrar la sesión**. Descárgalos o guárdalos en tu Google Drive.

In [ ]:
serie[['anio', 'ndvi', 'n_escenas']].to_csv('gredos_ndvi_verano_1985_2025.csv', index=False)

# Descargar al ordenador (solo funciona en Colab)
try:
    from google.colab import files
    files.download('gredos_ndvi_verano_1985_2025.csv')
except ImportError:
    print('Guardado en la carpeta de trabajo')

Para imágenes grandes, lo mejor es exportar a Google Drive desde Earth Engine. La tarea aparece en la pestaña *Tasks* del [Code Editor](https://code.earthengine.google.com/):

```python
tarea = ee.batch.Export.image.toDrive(
    image=pendiente, description='gredos_pendiente_sen', folder='curso_sig',
    region=cumbres, scale=30, crs='EPSG:25830', maxPixels=1e10)
tarea.start()
```

---

## Resumen

| Quiero... | Uso |
|---|---|
| Composites por periodo para varios años | `NdviSeasonality(...).get_year_composite()` |
| Un periodo de un año (rápido, en el servidor) | `get_period_composite(anio, periodo)` |
| Un GIF | `get_gif(name=..., bands=[...])` |
| Estadísticas por polígono | `imagen.reduceRegions(fc, reducer, scale)` |
| Serie de una zona, rápida | `map` + `reduceRegion` + **un solo** `getInfo()` |
| Tendencia de la serie | `TimeSeriesAnalyzer(...).analyze_trend(df, method='all')` |
| Tendencia por píxel | `ee.Reducer.sensSlope()` |

### Para el viernes, piensa en esto

- ¿Qué puede ver un satélite y qué no puede ver en una cumbre?
- Si el NDVI sube, ¿quiere decir "más especies", "más cobertura" o "más biomasa"?
- ¿Qué harías para que la tendencia no dependa del sensor, de la nieve ni del número de imágenes?